In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('train_and_test2.csv')

conn = sqlite3.connect('titanic.db')
df.to_sql('passengers', conn, 
          if_exists='replace', index=False)

print("Database created successfully!")
print(f"Rows loaded: {len(df)}")

Database created successfully!
Rows loaded: 1309


In [2]:
def query(sql):
    return pd.read_sql_query(sql, conn)

query("SELECT * FROM passengers LIMIT 5")

,Passengerid,Age,Fare,Sex,sibsp,zero,zero.1,zero.2,zero.3,zero.4,...,zero.12,zero.13,zero.14,Pclass,zero.15,zero.16,Embarked,zero.17,zero.18,2urvived
0,1,22.0,7.2500,0,1,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0
1,2,38.0,71.2833,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0.0,0,0,1
2,3,26.0,7.9250,1,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,1
3,4,35.0,53.1000,1,1,0,0,0,0,0,...,0,0,0,1,0,0,2.0,0,0,1
4,5,35.0,8.0500,0,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0


In [3]:
query("""
SELECT Sex, COUNT(*) as total,
       AVG(Age) as avg_age,
       AVG(Fare) as avg_fare
FROM passengers
GROUP BY Sex
""")

,Sex,total,avg_age,avg_fare
0,0,843,30.017888,26.140721
1,1,466,28.572082,46.198097


**## Step 3: Passenger Stats by Sex**

Group passengers by `Sex` to compare total count, average age, and average fare paid.

In [4]:
query("""
SELECT Pclass,
       COUNT(*) as passengers,
       SUM("2urvived") as survived,
       ROUND(AVG("2urvived") * 100, 1) as survival_rate
FROM passengers
GROUP BY Pclass
ORDER BY Pclass
""")

,Pclass,passengers,survived,survival_rate
0,1,323,136,42.1
1,2,277,87,31.4
2,3,709,119,16.8


**## Step 4: Survival Rate by Passenger Class**

Group passengers by `Pclass` to see how many survived in each class and calculate the survival rate (%).

In [5]:
query("""
SELECT 
  CASE 
    WHEN Age < 18 THEN 'Child'
    WHEN Age < 35 THEN 'YoungAdult'
    WHEN Age < 60 THEN 'Adult'
    ELSE 'Senior'
  END as age_group,
  COUNT(*) as count,
  ROUND(AVG("2urvived") * 100, 1) as survival_rate
FROM passengers
GROUP BY age_group
ORDER BY survival_rate DESC
""")

,age_group,count,survival_rate
0,Child,154,39.6
1,Adult,305,28.5
2,YoungAdult,810,23.1
3,Senior,40,17.5


**## Step 5: Survival Rate by Age Group**

Bucket passengers into age groups (Child, Young Adult, Adult, Senior) and compare survival rates across each group.

In [6]:
import pandas as pd, sqlite3
conn = sqlite3.connect('titanic.db')

port_data = pd.DataFrame({
    'port_code': [1.0, 2.0, 3.0],
    'port_name': ['Cherbourg','Queenstown','Southampton'],
    'country': ['France','Ireland','England']
})
port_data.to_sql('ports', conn,
    if_exists='replace', index=False)
print("Ports table created!")

Ports table created!


In [7]:
query = """
SELECT p.PassengerId, p.Age, p.Fare,
       p."2urvived",
       pt.port_name, pt.country
FROM passengers p
LEFT JOIN ports pt ON p.Embarked = pt.port_code
LIMIT 15
"""
pd.read_sql_query(query, conn)

,Passengerid,Age,Fare,2urvived,port_name,country
0,1,22.0,7.2500,0,Queenstown,Ireland
1,2,38.0,71.2833,1,NaN,NaN
2,3,26.0,7.9250,1,Queenstown,Ireland
3,4,35.0,53.1000,1,Queenstown,Ireland
4,5,35.0,8.0500,0,Queenstown,Ireland
5,6,28.0,8.4583,0,Cherbourg,France
6,7,54.0,51.8625,0,Queenstown,Ireland
7,8,2.0,21.0750,0,Queenstown,Ireland
8,9,27.0,11.1333,1,Queenstown,Ireland
9,10,14.0,30.0708,1,NaN,NaN


In [11]:
query = """
SELECT pt.country,
       COUNT(*) as passengers,
       ROUND(AVG(p."2urvived")*100,1) as survival_rate
FROM passengers p
LEFT JOIN ports pt ON p.Embarked = pt.port_code
GROUP BY pt.country
ORDER BY survival_rate DESC
"""
pd.read_sql_query(query, conn)

,country,passengers,survival_rate
0,NaN,272,34.9
1,France,123,24.4
2,Ireland,914,23.7


## Observation
Passengers with an unrecorded embarkation port had the highest 
survival rate at 34.9%, though this group of 272 passengers may 
include a mix of cabin classes that isn't captured by country alone.
Among known ports, France (Cherbourg) had the highest survival rate 
at 24.4%, followed closely by Ireland (Queenstown) at 23.7%. 
The difference between ports is relatively small, suggesting 
embarkation point alone is not a strong predictor of survival.

In [12]:
query = """
SELECT pt.port_name,
       ROUND(AVG(p.Fare),2) as avg_fare,
       COUNT(*) as total_passengers
FROM passengers p
INNER JOIN ports pt ON p.Embarked = pt.port_code
GROUP BY pt.port_name
ORDER BY avg_fare DESC
"""
pd.read_sql_query(query, conn)

,port_name,avg_fare,total_passengers
0,Queenstown,27.40,914
1,Cherbourg,12.41,123


## Observation
Passengers who boarded at Queenstown paid more than double the 
average fare of those from Cherbourg — £27.40 vs £12.41. This is 
unexpected, since Cherbourg was associated with more 1st class 
passengers historically. It's worth investigating passenger class 
distribution per port to understand this gap further.

In [13]:
query = """
SELECT COUNT(*) as missing_port
FROM passengers p
LEFT JOIN ports pt ON p.Embarked = pt.port_code
WHERE pt.port_name IS NULL
"""
pd.read_sql_query(query, conn)

,missing_port
0,272


## Observation
272 passengers (about 21% of the dataset) have no matching port 
record. This likely means their Embarked value doesn't correspond 
to any of the three port codes used, possibly representing missing 
or unrecorded data in the original dataset. This is an important 
data quality issue to flag before drawing strong conclusions from 
port-based analysis.